# 0. Configure your run

This is the only notebook you normally edit. Fill in the cells below, then run it
top to bottom. It saves two files in the project root:

* **`control_config.json`** — all your settings, read by `1_build`, `2_submit`
  and `3_sync` (so they always agree).
* **`jobs.json`** — the nested list of jobs to run.

Re-run this notebook whenever you change a setting.

## Project

Name, the notebooks to run in order, and what varies per job.

In [ ]:
project_name = "myproject"
notebooks = [
    "notebooks/0_settings.ipynb",   # first notebook: writes settings.json
    "notebooks/1_analysis.ipynb",
    "notebooks/2_more.ipynb",
]
varying = ["country", "region", "scenario"]   # the levels in jobs.json (below)

## The jobs to run

A nested dict: each root-to-leaf path is one SLURM job, and the levels line up
with `varying` above. nb2slurm builds the matching output folders and submits one
job per leaf.

**Format:** a *dict* nests one more level, a *list* at the bottom means several
jobs sharing that parent, and `None`/`[]` ends the path. It's plain JSON, so for
small runs type it by hand (below) and for big runs generate it in Python (a
comprehension, a CSV, an API query, ...) — see the README for the rules. The only
thing that matters is that the finished dict lands in `jobs.json`.

In [ ]:
jobs = {
    "NL": {"123": ["ssp126", "ssp245"]},
    "DE": {"789": ["ssp585"]},
}
# -> jobs: (NL,123,ssp126) (NL,123,ssp245) (DE,789,ssp585)

## HPC connection

Your cluster login details. The project dir is derived so it can't drift.

In [ ]:
username_on_slurm    = "me"
slurm_host           = "spider.surf.nl"
project_dir_on_slurm = f"/home/{username_on_slurm}/{project_name}"
ssh_key_file         = "~/.ssh/id_ed25519"

## Resources per job

What each SLURM job asks for, and where outputs go.

In [ ]:
time_for_job_to_run = "04:00:00"   # HH:MM:SS wall-clock limit
cpus_per_job        = 2
nodes_per_job       = 1
jobs_at_once        = 3            # max running in parallel
output_directory    = "output"    # could be "/scratch/me/output"

## Conda environment (optional)

Skip if your cluster already provides Python (set `conda_env`/`setup` instead).
Otherwise nb2slurm builds this environment + kernel for you in `1_build`.

*Need a different environment for one or two notebooks (e.g. a calibration step)?*
Add `kernels={"notebooks/step_8.ipynb": "myenv2"}` and
`extra_environments=[nb2slurm.Environment(name="myenv2", kernel="myenv2", ...)]`
to the `Workflow` in the **Save** cell below — see the README for details. The
single-environment setup here covers the common case.

In [ ]:
conda_env_name = "myenv"
kernel_name    = "myenv"
conda_packages = ["xarray", "numpy"]
pip_packages   = ["nb2slurm", "ewatercycle"]

## Data mounts (optional)

rclone mounts the job should set up before running.

In [ ]:
data_mounts = [
    {"remote": "dcache:/climate-data/caravan", "mountpoint": "/scratch/caravan"},
]

## Save

Builds the objects from your settings and writes `control_config.json` and
`jobs.json`. The other notebooks load these — you don't edit them by hand.

In [ ]:
import json, pathlib, nb2slurm

env = nb2slurm.Environment(
    name=conda_env_name, kernel=kernel_name,
    conda_packages=conda_packages, pip_packages=pip_packages,
)
wf = nb2slurm.Workflow(
    name=project_name, notebooks=notebooks, kernel=kernel_name,
    varying=varying, jobs_json="jobs.json",
    resources=dict(nodes=nodes_per_job, cpus=cpus_per_job, time=time_for_job_to_run),
    mounts=data_mounts, concurrency=jobs_at_once,
    output_dir=output_directory, environment=env,
)
cfg = nb2slurm.SSHConfig(
    host=slurm_host, user=username_on_slurm,
    remote_dir=project_dir_on_slurm, key_filename=ssh_key_file,
)

pathlib.Path("jobs.json").write_text(json.dumps(jobs, indent=2))
nb2slurm.save_config("control_config.json", workflow=wf, ssh=cfg)
print("saved control_config.json and jobs.json")